# MIDI Quality Check Notebook

Analyze generated MIDI files with objective quality metrics.

This notebook reports:
- Duration, note count, note density
- Pitch range and pitch-class spread
- Velocity and note-length stats
- Polyphony estimate
- Timing-grid consistency (rhythm regularity)
- Repetition indicator (n-gram repetition on pitches)

In [1]:
from __future__ import annotations

import math
from collections import Counter
from pathlib import Path

import pretty_midi

In [2]:
# --- Config ---
PROJECT_ROOT = Path("..").resolve()  # notebook in src/
GENERATED_DIR = PROJECT_ROOT / "generated"
TARGET_FILE = GENERATED_DIR / "sample_from_best.midi"  # can be .mid or .midi
TIME_STEP_SECONDS = 0.125

print("Generated dir exists:", GENERATED_DIR.exists())
print("Target file exists:", TARGET_FILE.exists())

Generated dir exists: True
Target file exists: True


In [3]:
def collect_notes(pm: pretty_midi.PrettyMIDI):
    notes = []
    for inst in pm.instruments:
        if inst.is_drum:
            continue
        notes.extend(inst.notes)
    notes.sort(key=lambda n: (n.start, n.end, n.pitch, n.velocity))
    return notes


def entropy_from_counts(counter: Counter) -> float:
    total = sum(counter.values())
    if total <= 0:
        return 0.0
    h = 0.0
    for c in counter.values():
        p = c / total
        h -= p * math.log2(max(p, 1e-12))
    return h


def pitch_ngram_repetition_ratio(pitches: list[int], n: int = 4) -> float:
    if len(pitches) < n:
        return 0.0
    ngrams = [tuple(pitches[i : i + n]) for i in range(len(pitches) - n + 1)]
    counts = Counter(ngrams)
    repeated = sum(c for c in counts.values() if c > 1)
    return repeated / max(1, len(ngrams))


def mean_polyphony(notes: list[pretty_midi.Note], time_step: float = 0.05) -> float:
    if not notes:
        return 0.0
    end_t = max(n.end for n in notes)
    if end_t <= 0:
        return 0.0

    t = 0.0
    values = []
    while t < end_t:
        active = 0
        for n in notes:
            if n.start <= t < n.end:
                active += 1
        values.append(active)
        t += time_step

    return sum(values) / max(1, len(values))

In [4]:
def analyze_midi(path: Path, grid_step: float = 0.125) -> dict:
    pm = pretty_midi.PrettyMIDI(str(path))
    notes = collect_notes(pm)
    if not notes:
        raise RuntimeError(f"No playable notes found in {path}")

    duration = float(pm.get_end_time())
    note_count = len(notes)
    note_density = note_count / max(duration, 1e-9)

    pitches = [n.pitch for n in notes]
    velocities = [n.velocity for n in notes]
    lengths = [max(1e-6, n.end - n.start) for n in notes]

    onset_steps = [round(n.start / grid_step) for n in notes]
    quantized_onsets = [s * grid_step for s in onset_steps]
    onset_err = [abs(n.start - q) for n, q in zip(notes, quantized_onsets)]

    pitch_class_counts = Counter(p % 12 for p in pitches)
    pitch_entropy = entropy_from_counts(Counter(pitches))
    pitch_class_entropy = entropy_from_counts(pitch_class_counts)

    # IOI (inter-onset interval) variability
    onset_times = sorted(n.start for n in notes)
    ioi = [max(1e-6, b - a) for a, b in zip(onset_times[:-1], onset_times[1:])]
    ioi_mean = sum(ioi) / len(ioi) if ioi else 0.0
    ioi_std = (sum((x - ioi_mean) ** 2 for x in ioi) / len(ioi)) ** 0.5 if ioi else 0.0

    repetition_ratio_4 = pitch_ngram_repetition_ratio(pitches, n=4)
    repetition_ratio_8 = pitch_ngram_repetition_ratio(pitches, n=8)

    return {
        "file": str(path),
        "duration_sec": duration,
        "note_count": note_count,
        "note_density_notes_per_sec": note_density,
        "pitch_min": min(pitches),
        "pitch_max": max(pitches),
        "pitch_range": max(pitches) - min(pitches),
        "pitch_entropy": pitch_entropy,
        "pitch_class_entropy": pitch_class_entropy,
        "velocity_min": min(velocities),
        "velocity_max": max(velocities),
        "velocity_mean": sum(velocities) / len(velocities),
        "note_len_mean_sec": sum(lengths) / len(lengths),
        "note_len_std_sec": (sum((x - (sum(lengths)/len(lengths)))**2 for x in lengths) / len(lengths)) ** 0.5,
        "mean_polyphony": mean_polyphony(notes),
        "grid_step_sec": grid_step,
        "mean_onset_grid_error_sec": sum(onset_err) / len(onset_err),
        "ioi_mean_sec": ioi_mean,
        "ioi_std_sec": ioi_std,
        "pitch_4gram_repetition_ratio": repetition_ratio_4,
        "pitch_8gram_repetition_ratio": repetition_ratio_8,
    }

In [5]:
report = analyze_midi(TARGET_FILE, grid_step=TIME_STEP_SECONDS)

print("MIDI Quality Report")
print("=" * 70)
for k, v in report.items():
    if isinstance(v, float):
        print(f"{k:32s}: {v:.6f}")
    else:
        print(f"{k:32s}: {v}")

MIDI Quality Report
file                            : C:\Users\eydos\Desktop\project x\Music_Generation\generated\sample_from_best.midi
duration_sec                    : 28.375000
note_count                      : 192
note_density_notes_per_sec      : 6.766520
pitch_min                       : 32
pitch_max                       : 94
pitch_range                     : 62
pitch_entropy                   : 5.081554
pitch_class_entropy             : 3.143976
velocity_min                    : 40
velocity_max                    : 103
velocity_mean                   : 72.380208
note_len_mean_sec               : 0.400391
note_len_std_sec                : 1.965049
mean_polyphony                  : 2.693662
grid_step_sec                   : 0.125000
mean_onset_grid_error_sec       : 0.000000
ioi_mean_sec                    : 0.147252
ioi_std_sec                     : 0.185790
pitch_4gram_repetition_ratio    : 0.010582
pitch_8gram_repetition_ratio    : 0.000000


## Quick Interpretation Guide

- **note_density_notes_per_sec**
  - very low (< 1): sparse/empty
  - very high (> 25): may sound cluttered
- **mean_polyphony**
  - near 1: mostly monophonic
  - 2 to 6: typical piano texture
  - very high: muddy chords/overlap
- **mean_onset_grid_error_sec**
  - closer to 0 means tighter rhythm on your quantization grid
- **pitch_4gram_repetition_ratio / pitch_8gram_repetition_ratio**
  - very high can indicate looping/repetitive output
- **pitch_range**
  - extremely narrow means little melodic movement; extremely wide may sound jumpy